# Lesson 1: Router Engine

Welcome to Lesson 1.This notebook is a light, simple update of the router engine example.It now uses current OpenAI model names and keeps the key code directly inside the notebook.

## Setup

In [ ]:
import osfrom dotenv import load_dotenvload_dotenv()OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")if not OPENAI_API_KEY:    raise ValueError("OPENAI_API_KEY not found in environment or .env file.")os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
import nest_asyncionest_asyncio.apply()

## Load Data

To download this paper, below is the needed code:#!wget "https://openreview.net/pdf?id=VtmBAGCN7o" -O metagpt.pdf**Note**: The pdf file is included with this lesson. To access it, go to the `File` menu and select`Open...`.

In [ ]:
from llama_index.core import SimpleDirectoryReader# load documentsdocuments = SimpleDirectoryReader(input_files=["metagpt.pdf"]).load_data()

## Define LLM and embedding model

In [ ]:
from llama_index.core.node_parser import SentenceSplittersplitter = SentenceSplitter(chunk_size=1024)nodes = splitter.get_nodes_from_documents(documents)

In [ ]:
from llama_index.core import Settingsfrom llama_index.llms.openai import OpenAIfrom llama_index.embeddings.openai import OpenAIEmbeddingLLM_MODEL = "gpt-5.4-nano"EMBED_MODEL = "text-embedding-3-small"Settings.llm = OpenAI(model=LLM_MODEL)Settings.embed_model = OpenAIEmbedding(model=EMBED_MODEL)print(f"LLM model: {LLM_MODEL}")print(f"Embedding model: {EMBED_MODEL}")

## Define Summary Index and Vector Index over the Same Data

In [ ]:
from llama_index.core import SummaryIndex, VectorStoreIndexsummary_index = SummaryIndex(nodes)vector_index = VectorStoreIndex(nodes)

## Define Query Engines and Set Metadata

In [ ]:
summary_query_engine = summary_index.as_query_engine(    response_mode="tree_summarize",    use_async=True,)vector_query_engine = vector_index.as_query_engine()

In [ ]:
from llama_index.core.tools import QueryEngineToolsummary_tool = QueryEngineTool.from_defaults(    query_engine=summary_query_engine,    description=(        "Useful for summarization questions related to MetaGPT"    ),)vector_tool = QueryEngineTool.from_defaults(    query_engine=vector_query_engine,    description=(        "Useful for retrieving specific context from the MetaGPT paper."    ),)

## Define Router Query Engine

In [ ]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEnginefrom llama_index.core.selectors import LLMSingleSelectorquery_engine = RouterQueryEngine(    selector=LLMSingleSelector.from_defaults(),    query_engine_tools=[        summary_tool,        vector_tool,    ],    verbose=True)

In [ ]:
response = query_engine.query("What is the summary of the document?")print(str(response))

In [ ]:
print(len(response.source_nodes))

In [ ]:
response = query_engine.query(    "How do agents share information with other agents?")print(str(response))

## Put everything together

In [ ]:
from llama_index.core import SimpleDirectoryReader, SummaryIndex, VectorStoreIndexfrom llama_index.core.node_parser import SentenceSplitterfrom llama_index.core.tools import QueryEngineToolfrom llama_index.core.query_engine.router_query_engine import RouterQueryEnginefrom llama_index.core.selectors import LLMSingleSelectordef get_router_query_engine(file_path: str, llm_model: str = LLM_MODEL, embed_model_name: str = EMBED_MODEL):    """Build a simple router query engine."""    llm = OpenAI(model=llm_model)    embed_model = OpenAIEmbedding(model=embed_model_name)    documents = SimpleDirectoryReader(input_files=[file_path]).load_data()    splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=100)    nodes = splitter.get_nodes_from_documents(documents)    summary_index = SummaryIndex(nodes)    vector_index = VectorStoreIndex(nodes, embed_model=embed_model)    summary_query_engine = summary_index.as_query_engine(        response_mode="tree_summarize",        use_async=True,        llm=llm,    )    vector_query_engine = vector_index.as_query_engine(llm=llm)    summary_tool = QueryEngineTool.from_defaults(        query_engine=summary_query_engine,        description="Useful for high-level summaries of the MetaGPT paper.",    )    vector_tool = QueryEngineTool.from_defaults(        query_engine=vector_query_engine,        description="Useful for finding specific details in the MetaGPT paper.",    )    return RouterQueryEngine(        selector=LLMSingleSelector.from_defaults(),        query_engine_tools=[summary_tool, vector_tool],        verbose=True,    )query_engine = get_router_query_engine("metagpt.pdf")

In [ ]:
response = query_engine.query("Tell me about the ablation study results.")print(str(response))